# Bài 4 — Mô phỏng kiến trúc Lambda
`bai4_lambda.ipynb`

---

Tích hợp kết quả Bài 2 (lớp tốc độ) và Bài 3 (lớp lô) thành **lớp phục vụ (serving layer)** thống nhất.

**Mục tiêu:**
- Hợp nhất kết quả từ batch và speed layer
- Xử lý đúng 3 kịch bản truy vấn
- Hiểu cơ chế chuyển giao (hand-off)

> ⚠️ Bài 2 và Bài 3 phải đã hoàn thành.

## Phần A — Thiết lập ranh giới batch/speed layer

In [ ]:
import os
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_OUTPUT = os.environ.get("DATA_OUTPUT_PATH", "/data/output")
DAILY_PATH  = f"{DATA_OUTPUT}/batch/daily_stats"
USER_PATH   = f"{DATA_OUTPUT}/batch/user_stats"
SPEED_PATH  = f"{DATA_OUTPUT}/anomalies"

spark = SparkSession.builder.appName("Bai4_Lambda").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Batch layer xử lý đến hết ngày hôm qua
BATCH_CUTOFF = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
print(f"BATCH_CUTOFF: {BATCH_CUTOFF}")

In [ ]:
# Kiểm tra dữ liệu đầu vào
def check_path(path, name):
    files = [f for _,_,fs in os.walk(path) for f in fs if f.endswith(".parquet")] if os.path.exists(path) else []
    print(f"{'✓' if files else '✗'} {name}: {len(files)} file parquet")
    return bool(files)

check_path(DAILY_PATH, "daily_stats (Bài 3)")
check_path(SPEED_PATH, "anomalies   (Bài 2)")
check_path(USER_PATH,  "user_stats  (Bài 3)")

## Phần B — Load dữ liệu từ hai lớp

**`load_batch_data(category, start_date, end_date)`**
- Đọc `daily_stats` với filter theo `date` và `category`
- Dùng `category="all"` để không filter theo category

**`load_speed_data(category)`**
- Đọc từ `/data/output/anomalies/`
- Nếu không có file, trả về DataFrame rỗng đúng schema

In [ ]:
def load_batch_data(category: str, start_date: datetime, end_date: datetime):
    # TODO: đọc DAILY_PATH, filter theo date range
    # TODO: nếu category != "all", filter thêm theo category
    pass


def load_speed_data(category: str, lookback_minutes: int = 60):
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
    empty_schema = StructType([
        StructField("category",      StringType()),
        StructField("total_revenue", DoubleType()),
        StructField("txn_count",     LongType()),
    ])
    # TODO: kiểm tra SPEED_PATH có file parquet không
    # Nếu không có: trả về spark.createDataFrame([], empty_schema)
    # Nếu có: đọc, filter lookback_minutes gần nhất, groupBy category
    pass


# Kiểm tra nhanh
bd = load_batch_data("electronics", datetime.now()-timedelta(days=10), datetime.now()-timedelta(days=1))
sd = load_speed_data("electronics")
print(f"Batch rows: {bd.count()}")
print(f"Speed rows: {sd.count()}")

## Phần C — Lớp phục vụ

Hàm `query_revenue(category, start_date, end_date)` trả về:
```python
{"total_revenue": float, "data_source": "batch"|"speed"|"combined", "coverage_note": str}
```

| Kịch bản | `data_source` |
|---|---|
| `end_date <= BATCH_CUTOFF` | `"batch"` |
| `start_date >= BATCH_CUTOFF` | `"speed"` |
| Vượt ranh giới | `"combined"` |

In [ ]:
def query_revenue(category: str, start_date: datetime, end_date: datetime) -> dict:
    # TODO: xác định kịch bản dựa trên BATCH_CUTOFF
    # TODO: kịch bản batch: load_batch_data, sum total_revenue
    # TODO: kịch bản speed: load_speed_data, sum total_revenue
    # TODO: kịch bản combined: batch (phần cũ) + speed (phần mới), cộng tổng
    # TODO: trả về dict với total_revenue, data_source, coverage_note
    pass


# Thử một truy vấn
print(query_revenue("electronics", datetime.now()-timedelta(days=5), datetime.now()-timedelta(days=1)))

## Phần D — Kiểm tra 3 kịch bản

In [ ]:
print("Kịch bản 1 — batch only:")
result_1 = query_revenue("electronics", datetime.now()-timedelta(days=10), datetime.now()-timedelta(days=5))
print(f"  {result_1}")
assert result_1["data_source"] == "batch"
print("  ✓\n")

print("Kịch bản 2 — speed only:")
result_2 = query_revenue("electronics", datetime.now()-timedelta(hours=1), datetime.now())
print(f"  {result_2}")
assert result_2["data_source"] == "speed"
print("  ✓\n")

print("Kịch bản 3 — combined:")
result_3 = query_revenue("electronics", datetime.now()-timedelta(days=3), datetime.now())
print(f"  {result_3}")
assert result_3["data_source"] == "combined"
print("  ✓")

## Phần E — Câu hỏi tư duy

**Câu 1:** Trong kịch bản `combined`, `total_revenue` có hoàn toàn chính xác không? Sự không chính xác đến từ đâu trong kiến trúc Lambda?

*(Trả lời tại đây)*

**Câu 2:** Khi batch job hoàn thành và `BATCH_CUTOFF` cập nhật, điều gì phải xảy ra với speed layer? Mô tả cơ chế hand-off.

*(Trả lời tại đây)*

**Câu 3:** Nếu chỉ dùng batch (không có speed layer), người dùng dashboard thấy gì khác biệt?

*(Trả lời tại đây)*

## Kiểm tra đầu ra

In [ ]:
print("=" * 45)
print("KIỂM TRA ĐẦU RA BÀI 4")
print("=" * 45)

checks = []
for label, kw in [("batch",datetime.now()-timedelta(days=10)),
                  ("speed",datetime.now()-timedelta(hours=1)),
                  ("combined",datetime.now()-timedelta(days=3))]:
    try:
        end = datetime.now()-timedelta(days=5) if label=="batch" else datetime.now()
        r = query_revenue("electronics", kw, end)
        checks.append((f"Kịch bản {label}: data_source đúng", r["data_source"]==label))
        checks.append((f"Kịch bản {label}: coverage_note không rỗng", len(r.get("coverage_note",""))>0))
    except Exception as e:
        checks.append((f"Kịch bản {label}", False))

all_passed = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    if not passed: all_passed = False
print()
print("✓ Bài 4 hoàn thành!" if all_passed else "✗ Xem lại phần chưa qua.")